In [22]:
# ! pip install torch transformers  datasets

Step 1: Import Dependencies


In [23]:
# Import necessary modules
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
import datasets

Step 2: Load Pre-trained Model

In [24]:
# Define the model checkpoint
model_checkpoint = "distilbert-base-uncased"

# Load the pre-trained tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)  # Adjust num_labels as needed

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step 3: Prepare the Dataset

In [25]:
# Load a sample dataset and split it into train and evaluation sets
from datasets import load_dataset
full_dataset = load_dataset("imdb", split="train[:500]")
dataset = full_dataset.train_test_split(test_size=0.2)  # 80% train, 20% evaluation

# Tokenize the datasets
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = {
    'train': dataset['train'].map(tokenize_function, batched=True),
    'eval': dataset['test'].map(tokenize_function, batched=True)
}


Step 4: Configure Training Parameters

In [26]:
# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    report_to="none", # Disable wandb and other reporting
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step 5: Create Data Collator and Metrics Function

In [27]:
# Define a data collator for padding
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Define a simple accuracy metric
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return {"accuracy": (predictions == labels).mean()}

Step 6: Create Processing Class

In [28]:
# Create a processing class
class SimpleProcessor:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, text, *args, **kwargs):
        return self.tokenizer(text, *args, **kwargs)

    # Add the save_pretrained method to SimpleProcessor
    def save_pretrained(self, save_directory):
        self.tokenizer.save_pretrained(save_directory)

processor = SimpleProcessor(tokenizer)

Step 7: Initialize Trainer and Start Training

In [29]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['eval'],
    processing_class=processor,  # Use processing_class instead of tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

# Start the fine-tuning process
trainer.train()

c:\Users\HK\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.000175,1.000000
2,No log,0.000065,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\HK\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=200, training_loss=0.01225612998008728, metrics={'train_runtime': 3011.3536, 'train_samples_per_second': 0.398, 'train_steps_per_second': 0.1, 'total_flos': 105973918924800.0, 'train_loss': 0.01225612998008728, 'epoch': 2.0})

Step 8: Save the Model

In [30]:
# Save the fine-tuned model
model_path = "./fine_tuned_model"
trainer.save_model(model_path)
tokenizer.save_pretrained(model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine_tuned_model\\tokenizer_config.json',
 './fine_tuned_model\\tokenizer.json')

Step 9: Test the Model


In [31]:
# Test the model with a simple example
test_input = "The movie was absolutely fantastic and I loved it."
test_encoding = tokenizer(test_input, return_tensors="pt")

# Get model predictions
with torch.no_grad():
    logits = model(**test_encoding).logits

# Convert logits to predicted class
predicted_class = torch.argmax(logits, dim=1)
print("Predicted class:", predicted_class.item())
print("Class meaning:", "Positive" if predicted_class.item() == 1 else "Negative")

Predicted class: 0
Class meaning: Negative


In [ ]:
#Process Findings: 
#During the implementation of the fine-tuning pipeline using the Hugging Face Transformers library, several issues were initially encountered. The first issue involved dependency installation and environment setup, particularly when using Git Bash on Windows. Commands such as `pip install` and virtual environment activation failed because Python and pip were not correctly configured in the system PATH, and Windows command syntax was mixed with Git Bash syntax. Another issue emerged during the training configuration phase, where incorrect parameter names and dataset formatting inconsistencies caused execution errors. Additionally, the Trainer API required a compatible processing object and correctly tokenized datasets to function properly during training and evaluation.
# To resolve these issues, multiple prompts were used with the LLM to diagnose environment setup problems, correct syntax errors, and refine the Hugging Face training pipeline. The most useful suggestions included using `python -m pip install` instead of `pip install`, activating the virtual environment with `source venv/Scripts/activate` in Git Bash, and ensuring the tokenizer outputs were compatible with the Trainer API. The LLM also recommended creating a custom `SimpleProcessor` class with a `save_pretrained()` method so the Trainer could properly manage tokenizer saving operations. Furthermore, the training configuration was refined by setting evaluation and save strategies to `"epoch"`, disabling unnecessary logging integrations with `report_to="none"`, and adding `EarlyStoppingCallback` to prevent overfitting. These modifications ensured that the training loop executed correctly and that the model could be saved and reloaded without errors.
# Overall, the debugging process demonstrated how advanced prompt engineering techniques can significantly improve development efficiency when working with complex ML frameworks. Carefully structured prompts that included the full error message, code context, and execution environment produced more accurate and actionable responses from the LLM. One remaining challenge was ensuring compatibility between different library versions, especially across Transformers, Datasets, and PyTorch. However, the experience highlighted that LLM-assisted debugging is highly effective for identifying configuration issues, correcting implementation mistakes, and accelerating experimentation in machine learning workflows.
